# 실습 3 · 그리드월드와 Q러닝

**인공지능 일반 · 5차시**

지난 시간에 우리는 3×3 격자에서 Q표를 **손으로** 채웠습니다.
오늘은 그 계산을 **코드가 대신하게** 만들고, 격자를 5×5로 키웁니다.

| 순서 | 내용 |
|---|---|
| 1 | 환경 만들기 — 5×5 그리드월드 |
| 2 | Q표 만들기 |
| 3 | **ε-그리디** — 어느 방향으로 갈까 |
| 4 | **Q 업데이트** — 오늘의 핵심 한 줄 |
| 5 | 학습 루프 — 500판 돌리기 |
| 6 | 결과 보기 — 가치 지도와 정책 화살표 |
| 7 | 손잡이 돌려보기 — ε, γ, α |

> 셀 실행은 `Shift + Enter`. 위에서부터 **순서대로** 실행하세요.

## ✏️ 이 노트북은 실습용입니다

코드 중간중간 `여기를_채우세요` 로 표시된 **빈칸 5곳**이 있습니다. 주석의 힌트를 읽고 채운 뒤 셀을 실행하세요.

빈칸을 채우지 않고 실행하면 `NameError: name '여기를_채우세요' is not defined` 에러가 납니다 — **"아직 못 채웠다"는 뜻**입니다.

| 번호 | 위치 | 무엇을 채우나요 |
|---|---|---|
| ① | Q표 만들기 | 0으로 가득 찬 (상태 × 행동) 표 |
| ② | ε-그리디 | 탐험할 때 방향 아무거나 고르기 |
| ③ | Q 업데이트 | **벨만 목표값** `r + γ·max Q(s′,·)` |
| ④ | Q 업데이트 | **새 Q값** `Q + α(목표 − Q)` |
| ⑤ | 학습 루프 | 고른다 → 겪는다 → 고친다 조립하기 |

③번과 ④번이 4차시에 손으로 계산했던 바로 그 두 칸입니다.

## 0. 준비

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

# --- 그래프에 한글을 쓰기 위한 준비 (실패해도 실습은 진행됩니다) ---
try:
    import matplotlib.font_manager as fm
    !apt-get -qq install fonts-nanum > /dev/null 2>&1
    fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    plt.rcParams['font.family'] = 'NanumGothic'
except Exception as e:
    print('한글 폰트 설정 실패 — 그래프의 한글이 □□로 보일 수 있습니다.')

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

print('준비 완료!')

## 1. 환경 만들기 — 5×5 그리드월드

지도를 글자로 그립니다.

| 글자 | 뜻 | 보상 |
|---|---|---|
| `S` | 출발점 | – |
| `.` | 빈 칸 | 0 |
| `#` | 벽 (못 지나감, 부딪히면 제자리) | 0 |
| `X` | 구덩이 — **판 끝** | **−1** |
| `o` | **작은 보물** — 판 끝 | **+0.2** |
| `G` | 진짜 목표 — **판 끝** | **+1** |

작은 보물 `o`는 출발점에서 **4걸음**, 진짜 목표 `G`는 **8걸음**입니다.
가깝지만 적은 것과, 멀지만 큰 것 — 3차시 밴딧에서 본 그 딜레마가 여기서 다시 나옵니다.

**중요:** 우리가 만들 AI는 이 지도를 **보지 못합니다.**
오직 `step()`이 돌려주는 `(도착 칸, 보상, 끝났나)` 세 값만 보고 배웁니다.

In [ ]:
GRID = [
    "S...o",
    ".##.X",
    ".....",
    "X.##.",
    "....G",
]

ROWS, COLS = len(GRID), len(GRID[0])
N_STATES   = ROWS * COLS          # 25개 칸
N_ACTIONS  = 4                    # ↑ ↓ ← →
ACTIONS    = ['↑', '↓', '←', '→']
MOVES      = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # (행 변화, 열 변화)

def to_state(r, c):  return r * COLS + c          # (행, 열) → 칸 번호
def to_rc(s):        return divmod(s, COLS)       # 칸 번호 → (행, 열)

START = [to_state(r, c) for r in range(ROWS) for c in range(COLS) if GRID[r][c] == 'S'][0]
GOAL  = [to_state(r, c) for r in range(ROWS) for c in range(COLS) if GRID[r][c] == 'G'][0]


def reset():
    '''새 판 시작 → 출발 칸 번호'''
    return START


def step(s, a):
    '''s칸에서 a방향으로 한 칸.  →  (도착 칸 s2, 보상 r, 끝났나 done)'''
    r0, c0 = to_rc(s)
    dr, dc = MOVES[a]
    r1, c1 = r0 + dr, c0 + dc

    # 격자 밖이거나 벽이면 → 제자리
    if not (0 <= r1 < ROWS and 0 <= c1 < COLS) or GRID[r1][c1] == '#':
        r1, c1 = r0, c0

    s2 = to_state(r1, c1)
    ch = GRID[r1][c1]

    if ch == 'G':  return s2,  1.0, True      # 진짜 목표 → 판 끝
    if ch == 'o':  return s2,  0.2, True      # 작은 보물 → 판 끝
    if ch == 'X':  return s2, -1.0, True      # 구덩이 → 판 끝
    return s2, 0.0, False                     # 그 외 → 보상 0, 계속


def argmax_random(v):
    '''가장 큰 값의 번호. 동점이면 그중 하나를 무작위로.
       (3차시 톰슨 샘플링에서 쓴 그 함수입니다)'''
    m = np.max(v)
    cands = [i for i in range(len(v)) if v[i] == m]
    return random.choice(cands)


def show_map(pos=None):
    '''지도를 글자로 출력. pos에 칸 번호를 주면 그 자리에 로봇을 그린다.'''
    for r in range(ROWS):
        line = ''
        for c in range(COLS):
            ch = GRID[r][c]
            if pos is not None and to_state(r, c) == pos:
                line += ' @ '        # @ = 지금 로봇이 서 있는 칸
            elif ch == '#':  line += ' ■ '
            elif ch == 'X':  line += ' X '
            elif ch == 'o':  line += ' o '
            elif ch == 'G':  line += ' G '
            else:            line += ' . '
        print(line)

print(f'칸 {N_STATES}개 × 방향 {N_ACTIONS}개 = Q표 칸 {N_STATES * N_ACTIONS}개')
print(f'출발 s{START}, 목표 s{GOAL} (작은 보물은 4걸음 거리)\n')
show_map(START)

### 손으로 몇 걸음 움직여 보기

AI에게 맡기기 전에, `step()`이 어떻게 돌아가는지 직접 확인합시다.
아래 `plan`의 숫자를 바꿔가며 실행해 보세요. (0=↑, 1=↓, 2=←, 3=→)

In [ ]:
plan = [1, 1, 3, 1, 1, 3, 3, 3]   # ← 이 방향들로 차례차례 움직인다

s = reset()
print(f'출발: s{s}')
for t, a in enumerate(plan, 1):
    s2, r, done = step(s, a)
    mark = ''
    if s2 == s:  mark = '  ← 벽! 제자리'
    if done:     mark = '  ← 판 끝!'
    print(f'{t:2d}걸음  {ACTIONS[a]}   s{s} → s{s2}   보상 {r:g}{mark}')
    s = s2
    if done:
        break

print()
show_map(s)

## 2. Q표 만들기

4차시의 그 표입니다. 이번엔 **25행 × 4열**.
처음에는 **전부 0** — AI는 아무것도 모르는 상태로 시작합니다.

In [ ]:
# 실습 ① : 0으로 가득 찬 (상태 × 행동) 표를 만든다
Q = 여기를_채우세요        # 힌트: np.zeros((행 개수, 열 개수))

print('Q표 크기:', Q.shape)
print('전부 0인가?', np.all(Q == 0))
print()
print('s0(출발 칸)의 네 방향 Q값:', Q[START])

## 3. ε-그리디 — 어느 방향으로 갈까

- 확률 `ε` : 아무 방향이나 (**탐험**)
- 확률 `1−ε` : Q값이 가장 큰 방향 (**활용**)

3차시 밴딧에서 쓴 것과 **똑같습니다.** 달라진 건 `Q[s]` 처럼 **상태를 먼저 찾아 들어간다**는 것뿐.

In [ ]:
def choose_action(Q, s, eps):
    '''s칸에서 방향 하나 고르기'''
    if random.random() < eps:
        # 실습 ② : 탐험 — 네 방향 중 아무거나
        return 여기를_채우세요     # ✏️ 실습 ② : 0 ~ 3 중 아무 방향이나 (힌트: random.randrange)
    return argmax_random(Q[s])        # 활용 — Q값 1등


# 확인: 표가 전부 0이면 (=아는 게 없으면) 방향이 골고루 나와야 한다
random.seed(0)
counts = [0, 0, 0, 0]
for _ in range(2000):
    counts[choose_action(Q, START, eps=0.0)] += 1
print('eps=0 인데도 골고루 나옵니다 (전부 동점이라 argmax_random이 무작위로 고름)')
print({ACTIONS[i]: counts[i] for i in range(4)})

## 4. Q 업데이트 — 오늘의 핵심 한 줄

4차시에 손으로 계산했던 그 두 칸입니다.

$$\text{목표} = r + \gamma \cdot \max_{a'} Q(s', a')$$
$$Q(s,a) \leftarrow Q(s,a) + \alpha\,(\text{목표} - Q(s,a))$$

**잊지 마세요:** 판이 끝났으면(`done`) 이어질 미래가 없으므로 목표는 그냥 `r` 입니다.

In [ ]:
def q_update(Q, s, a, r, s2, done, alpha, gamma):
    '''Q표의 한 칸을 고친다 (표를 제자리에서 직접 수정)'''
    if done:
        target = r                      # 종료 칸 → 미래 없음
    else:
        # 실습 ③ : 벨만 목표값
        target = 여기를_채우세요   # ✏️ 실습 ③ : r + γ × (도착 칸 s2의 최고 Q값)  힌트: np.max(Q[s2])

    # 실습 ④ : 새 Q값
    Q[s, a] = 여기를_채우세요      # ✏️ 실습 ④ : 현재값 + α × (목표 − 현재값)

### 4차시에 손으로 푼 문제로 검증하기

지난 시간 활동의 상황 그대로입니다.

> 어떤 칸에서 → 로 갔더니, 보상은 0이고, **도착한 칸의 최고 Q값이 1.0** 이었다.
> α=1, γ=0.9 라면 새 Q값은?  → `0 + 0.9 × 1.0 = 0.9`

아래 셀이 **에러 없이** 지나가면 ③④번을 제대로 채운 겁니다.

In [ ]:
testQ = np.zeros((3, 4))
testQ[1, 3] = 1.0          # 1번 칸의 → 방향 Q값이 1.0 이라고 치자

# 0번 칸에서 →(3번 행동)로 가서 1번 칸에 도착, 보상 0, 아직 안 끝남
q_update(testQ, s=0, a=3, r=0.0, s2=1, done=False, alpha=1.0, gamma=0.9)
assert abs(testQ[0, 3] - 0.9) < 1e-9, f'0.9가 나와야 하는데 {testQ[0, 3]} 이 나왔습니다'
print(f'검증 1 통과 :  0 + 0.9 × 1.0 = {testQ[0, 3]}')

# 이번엔 종료 칸(구덩이)에 빠진 경우 → 목표는 그냥 -1
q_update(testQ, s=2, a=1, r=-1.0, s2=99, done=True, alpha=1.0, gamma=0.9)
assert abs(testQ[2, 1] + 1.0) < 1e-9
print(f'검증 2 통과 :  종료 칸이므로 목표 = r = {testQ[2, 1]}')

# α=0.5 라면 절반만 반영된다
testQ2 = np.zeros((2, 4)); testQ2[1, 0] = 1.0
q_update(testQ2, s=0, a=0, r=0.0, s2=1, done=False, alpha=0.5, gamma=0.9)
assert abs(testQ2[0, 0] - 0.45) < 1e-9
print(f'검증 3 통과 :  0 + 0.5 × (0.9 − 0) = {testQ2[0, 0]}')
print('\n세 개 다 통과했습니다. 이제 진짜로 학습시켜 봅시다.')

## 5. 학습 루프 — 500판 돌리기

4차시 마지막에 본 의사코드 그대로입니다.

```
에피소드를 반복:
    s ← 출발 칸
    판이 끝날 때까지:
        고른다 → 겪는다 → 고친다 → 넘어간다
```

`eps`는 **1.0에서 시작해 0.05까지 서서히 줄입니다**(ε-decay).
처음엔 마음껏 헤매고, 나중엔 아는 길로 가라는 뜻입니다.

In [ ]:
def train(episodes=1000, alpha=0.5, gamma=0.95,
          eps_start=1.0, eps_end=0.05, max_steps=100, seed=0):

    random.seed(seed); np.random.seed(seed)
    Q = np.zeros((N_STATES, N_ACTIONS))       # 빈 표에서 시작
    returns, lengths = [], []                 # 판마다 점수 / 걸음 수 기록

    for ep in range(episodes):
        # eps를 서서히 줄인다 (앞쪽 70% 구간 동안 직선으로 감소)
        eps = max(eps_end, eps_start - (eps_start - eps_end) * ep / (episodes * 0.7))

        s = reset()
        total = 0.0
        for t in range(max_steps):
            # ✏️ 실습 ⑤ : 네 줄을 채워 한 걸음을 완성하세요
            a  = 여기를_채우세요                       # 고른다  (choose_action)
            s2, r, done = 여기를_채우세요              # 겪는다  (step)
            여기를_채우세요                            # 고친다  (q_update)
            s  = 여기를_채우세요                       # 다음 칸으로 넘어간다

            total += r
            if done:
                break

        returns.append(total)
        lengths.append(t + 1)

    return Q, returns, lengths


Q, returns, lengths = train()
print(f'마지막 100판 평균 점수 : {np.mean(returns[-100:]):+.2f}')
print(f'   1.0 = 진짜 목표(G)에 도달  ·  0.2 = 작은 보물(o)에 만족  ·  −1.0 = 구덩이')
print(f'마지막 100판 평균 걸음 : {np.mean(lengths[-100:]):.1f}     (G까지 최단 경로는 8걸음)')

### 학습 곡선

**왼쪽**은 판마다 받은 점수를 50판씩 묶어 평균 낸 것, **오른쪽**은 판이 끝날 때까지 걸린 걸음 수입니다.

점수가 **0.2 근처에서 한참 머물다가 1.0으로 올라가는** 구간이 보이면,
그건 AI가 *작은 보물에 만족하다가 진짜 목표를 발견한 순간*입니다.

In [ ]:
def smooth(x, k=50):
    x = np.array(x, dtype=float)
    return np.convolve(x, np.ones(k) / k, mode='valid')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.2))

ax1.plot(smooth(returns), color='#0e8a52', lw=2)
ax1.axhline(1.0, color='#16233c', ls=':', lw=1.2)
ax1.set_title('판당 점수 (50판 이동평균)'); ax1.set_xlabel('에피소드'); ax1.set_ylim(-1.1, 1.2)

ax2.plot(smooth(lengths), color='#0a7ec2', lw=2)
ax2.axhline(8, color='#c8324a', ls=':', lw=1.2)
ax2.set_title('목표까지 걸린 걸음 수 (빨간 점선 = 최단 8걸음)'); ax2.set_xlabel('에피소드')

plt.tight_layout(); plt.show()

print('처음엔 구덩이에 빠지거나 헤매다가, 점점 +1을 8걸음에 받아냅니다.')

## 6. 결과 보기 — 표가 지도가 되었나

- **가치 지도** : 각 칸의 `max Q(s, ·)` — "여기서부터 잘하면 얼마 받나"
- **정책 화살표** : 각 칸에서 Q값이 가장 큰 방향 — 곧 **길**

In [ ]:
def show_result(Q, title=''):
    V = Q.max(axis=1).reshape(ROWS, COLS)          # 칸마다 최고 Q값
    mask = np.zeros((ROWS, COLS), dtype=bool)

    fig, ax = plt.subplots(figsize=(5.2, 5.2))
    im = ax.imshow(V, cmap='YlGn', vmin=min(V.min(), 0), vmax=max(V.max(), 1))

    for r in range(ROWS):
        for c in range(COLS):
            ch, s = GRID[r][c], to_state(r, c)
            if ch == '#':
                ax.add_patch(plt.Rectangle((c - .5, r - .5), 1, 1, color='#5d6c86'))
                continue
            if ch in 'GXo':
                col = {'G': '#0e8a52', 'X': '#c8324a', 'o': '#b06a00'}[ch]
                ax.text(c, r, ch, ha='center', va='center',
                        fontsize=22, fontweight='bold', color=col)
                continue
            # 가치 숫자
            ax.text(c, r + .34, f'{V[r, c]:.2f}', ha='center', va='center',
                    fontsize=8, color='#5d6c86')
            # 최선의 방향을 화살표로
            dr, dc = MOVES[int(np.argmax(Q[s]))]
            ax.arrow(c - dc * .18, r - dr * .18, dc * .34, dr * .34,
                     head_width=.16, head_length=.13, fc='#16233c', ec='#16233c', lw=1.4)
            if s == START:
                ax.add_patch(plt.Rectangle((c - .5, r - .5), 1, 1, fill=False,
                                           ec='#0a7ec2', lw=3))

    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title or '가치 지도와 정책 (파란 테두리 = 출발)')
    plt.colorbar(im, ax=ax, shrink=.8, label='max Q(s, ·)')
    plt.tight_layout(); plt.show()


show_result(Q)

**읽는 법**

- 목표(G) 근처가 **진하고**, 멀어질수록 **옅어집니다** — γ를 계속 곱했으니까요
- 구덩이(X) 옆 칸의 화살표는 **구덩이를 피해** 있습니다
- 작은 보물(o) 쪽 값도 0보다는 크지만, **G쪽 길보다 낮습니다**. 화살표가 o를 지나쳐 갑니다
- 화살표를 출발점부터 따라가 보세요. **길이 보입니다**

이 화살표들은 아무도 그려 준 적이 없습니다. **보상 두 개(+1, −1)만 주고 500판 돌린 결과**입니다.

In [ ]:
def play(Q, verbose=True):
    '''학습된 Q표로 한 판 (탐험 없이 Q값 1등만 따라감)'''
    s = reset(); path = [s]; total = 0.0
    for t in range(100):
        a = int(np.argmax(Q[s]))
        s, r, done = step(s, a)
        path.append(s); total += r
        if verbose:
            print(f'{t+1:2d}걸음  {ACTIONS[a]}  → s{s}')
        if done:
            break
    if verbose:
        print(f'\n{t+1}걸음 만에 종료, 점수 {total:+.0f}\n')
        show_map(s)
    return total, t + 1

play(Q);

## 7. 손잡이 돌려보기

α, γ, ε — 4차시에 이야기한 세 손잡이를 실제로 돌려 봅시다.

In [ ]:
# ── ε : 탐험을 아예 안 하면? ──
settings = {
    'ε=0 (탐험 없음)'      : dict(eps_start=0.0, eps_end=0.0),
    'ε=0.1 고정'           : dict(eps_start=0.1, eps_end=0.1),
    'ε 1.0 → 0.05 (감소)'  : dict(eps_start=1.0, eps_end=0.05),
}

plt.figure(figsize=(7.5, 3.2))
for name, kw in settings.items():
    scores = []
    for seed in range(5):                      # 씨앗 5개 평균 (운 지우기)
        Qx, ret, _ = train(seed=seed, **kw)
        scores.append(smooth(ret, 50))
    plt.plot(np.mean(scores, axis=0), lw=2, label=name)

plt.axhline(1.0, color='#16233c', ls=':', lw=1)
plt.title('탐험을 얼마나 할 것인가'); plt.xlabel('에피소드'); plt.ylabel('판당 점수')
plt.legend(); plt.ylim(-1.1, 1.2); plt.tight_layout(); plt.show()

for name, kw in settings.items():
    finals = [np.mean(train(seed=s, **kw)[1][-100:]) for s in range(5)]
    print(f'{name:<22} 마지막 100판 평균 점수 {np.mean(finals):+.2f}')

print()
print('ε=0 은 처음 찾은 작은 보물(+0.2)에 갇혀 나오지 못합니다 — 3차시 그리디의 함정 그대로입니다.')
print('가 본 적 없는 길의 Q값은 0이라, 0.2짜리 길이 늘 이겨 버리니까요.')

In [ ]:
# ── γ : 얼마나 멀리 볼 것인가 ──
print('γ        출발 칸의 값    학습 후 경로     G 도달률')
print('-' * 46)
for g in [0.5, 0.9, 0.95, 0.99, 1.0]:
    lens, wins, v0 = [], [], []
    for seed in range(5):                       # 씨앗 5개 평균
        Qg, _, _ = train(gamma=g, seed=seed)
        total, steps = play(Qg, verbose=False)
        lens.append(steps); wins.append(total > 0.5); v0.append(Qg[START].max())   # 0.2(작은 보물) 말고 진짜 G만
    print(f'{g:<9}{np.mean(v0):8.3f}{np.mean(lens):13.1f}걸음{np.mean(wins)*100:10.0f}%')

print()
print('γ=0.5 : 출발 칸의 값이 거의 0 — 8걸음 거리의 +1이 여기까지 닿지 못합니다.')
print('        그래서 4걸음짜리 작은 보물(+0.2)에 만족합니다. 근시안적인 AI입니다.')
print('γ=1.0 : 돌아가는 길과 지름길의 값이 똑같이 1.0 — 최단 경로를 고집할 이유가 없어져')
print('        100걸음을 헤매다 시간이 끝납니다. γ<1이 곧 "빨리 가라"는 압력이었던 겁니다.')
print()
print('γ는 "얼마나 멀리 보는가"입니다. 너무 가까이 보면 눈앞의 것에 만족하고,')
print('무한히 멀리 보면 서두를 이유가 사라집니다.')

show_result(train(gamma=0.5)[0], title='γ = 0.5 — 가까운 작은 보물로 화살표가 몰린다')

---

## 도전 과제

1. **α를 바꿔 보세요.** `train(alpha=1.0)` / `train(alpha=0.1)`
   우리 격자는 미끄러지지 않으므로 α=1.0 이어도 정확히 배웁니다. 대신 **학습 속도**가 어떻게 달라지나요?

2. **지도를 직접 그려 보세요.** `GRID`의 글자를 바꾸면 그대로 새 환경이 됩니다.
   벽으로 막힌 방을 만들거나, 구덩이를 늘려 보세요. **목표가 도달 불가능하면** 어떻게 되나요?

   `step()`에서 작은 보물의 보상을 `0.2` 대신 **`0.8`**로 올려 보세요.
   ε을 아무리 잘 조절해도 AI가 G로 가지 않는 지점이 있습니다. 그 값은 얼마쯤인가요?
   (힌트: G의 가치는 출발 기준 0.95⁷ ≈ 0.70입니다)

3. **걸음마다 −0.01점씩 주어 보세요.** `step()`에서 마지막 `return s2, 0.0, False` 를
   `return s2, -0.01, False` 로 바꿉니다. γ=1.0 으로 두어도 최단 경로를 찾아내나요? 왜일까요?

4. **미끄러지는 빙판을 만들어 보세요.** `step()` 첫 줄에
   `if random.random() < 0.2: a = random.randrange(4)` 를 넣으면 20% 확률로 엉뚱한 방향으로 미끄러집니다.
   이때 α=1.0 과 α=0.1 중 어느 쪽이 안정적인가요? **4차시에서 예고한 이야기입니다.**

5. **Q표를 눈으로 확인해 보세요.**
   ```python
   import pandas as pd
   pd.DataFrame(Q, columns=ACTIONS).round(3)
   ```
   4차시에 손으로 채운 그 표가, 이번엔 25줄짜리로 완성되어 있습니다.

6. **여기까지가 '표'의 한계입니다.** 우리 표는 25×4 = 100칸이었습니다.
   바둑처럼 상태가 10¹⁷⁰개라면 이 표는 아예 만들 수 없습니다.
   그때는 표 대신 **신경망**에게 Q값을 계산하게 시키고, 그 방법의 이름이 **DQN**입니다.
   벨만 방정식도 ε-그리디도 오늘 것 그대로이고 **표가 신경망으로 바뀔 뿐**이지만,
   그 앞은 딥러닝의 영역입니다. 이런 길이 있다는 것만 알아 두세요.